# 修了課題②　MNIST

利用するデータセット：MNIST

合格基準：テストデータに対して正解率95%以上

# データセットのダウンロード


In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

In [3]:
# MNISTデータのロードと訓練データ、検証データの分割
def load_mnist():
    mnist = fetch_openml('MNIST_784', version=1, data_home="MNIST_data/")
    mnist_X, mnist_y = shuffle(mnist.data[:60000].astype('float32'),
                               mnist.target[:60000].astype('int32'), random_state=42)

    # 正規化
    mnist_X = mnist_X / 255.0

    # one-hot表現に変換
    mnist_y = np.eye(10)[mnist_y]

    return train_test_split(mnist_X, mnist_y,
                test_size=0.2,
                random_state=42)

In [4]:
# それぞれのデータの形式を出力
x_train, x_val, y_train, y_val = load_mnist()
print('学習データ ', x_train.shape)
print('正解ラベル ', y_train.shape)
print('検証データ ', x_val.shape)
print('正解ラベル ', y_val.shape)

学習データ  (48000, 784)
正解ラベル  (48000, 10)
検証データ  (12000, 784)
正解ラベル  (12000, 10)


# モデルの制作

In [5]:
# ここからダウンロードされたデータに対してモデルを作成し、検証用データに対して予測結果を出力して下さい。

# 提出形式

## 必要なライブラリーのインポート

In [6]:
import os
# TPUを利用する場合
if 'COLAB_TPU_ADDR' in os.environ:
    import jax
    import jax.numpy as np
    print("Using TPU with JAX")

# GPUを利用する場合
elif os.system("nvidia-smi > /dev/null 2>&1") == 0:
    import cupy as np
    print("Using GPU with CuPy")

# CPUを利用する場合
else:
    import numpy as np
    print("Using CPU with NumPy")

import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
from collections import OrderedDict
import pickle

Using CPU with NumPy


#1.データのダウンロードと成形

## データのダウンロード

今回はDeepConvNetの入力形式に合わせるため、

正規化を行った後にそれぞれの画像の次元に(28,28)の次元を追加しています。

In [7]:
# MNISTデータのロードと訓練データ、テストデータの分割を行う
def load_mnist():
    mnist = fetch_openml('MNIST_784', version=1, data_home="MNIST_data/")
    mnist_X, mnist_y = shuffle(mnist.data[:60000].astype('float32'),
                               mnist.target[:60000].astype('int32'), random_state=42)

    # 正規化を行う
    mnist_X = mnist_X / 255.0
    mnist_X = np.array(mnist_X)
    train_images = []
    #次元を追加する
    for i in range(mnist_X.shape[0]):
        train_images.append(np.expand_dims(mnist_X[i].reshape(28, 28), axis=0))
    mnist_X = np.array(train_images)

    # one-hot表現に変換する
    mnist_y = np.eye(10)[np.array(mnist_y)]

    return train_test_split(mnist_X, mnist_y,
                test_size=0.2,
                random_state=42)

In [8]:
def load_mnist_from_csv(train_csv_path, test_csv_path):
    # CSVファイルの読み込み
    train_df = pd.read_csv(train_csv_path)
    test_df = pd.read_csv(test_csv_path)

    # データとラベルの分割
    train_X = np.array(train_df.iloc[:, 1:].values.astype('float32')) / 255.0
    train_y = np.array(train_df.iloc[:, 0].values.astype('int32'))

    test_X = np.array(test_df.iloc[:, 1:].values.astype('float32')) / 255.0
    test_y = np.array(test_df.iloc[:, 0].values.astype('int32'))

    # 次元の追加 (28x28に変換し、チャンネル次元を追加)
    train_X = np.array([np.expand_dims(img.reshape(28, 28), axis=0) for img in train_X])
    test_X = np.array([np.expand_dims(img.reshape(28, 28), axis=0) for img in test_X])

    # ラベルをone-hot表現に変換
    train_y = np.eye(10)[train_y]
    test_y = np.eye(10)[test_y]

    return train_X, train_y, test_X, test_y

In [9]:
# それぞれのデータの形式を出力する
x_train, x_val, y_train, y_val = load_mnist()
# x_train, y_train, x_val, y_val = load_mnist_from_csv('mnist_train.csv', 'mnist_test.csv')
print('学習データ ', x_train.shape)
print('学習データの正解ラベル ', y_train.shape)
print('検証データ ', x_val.shape)
print('検証データの正解ラベル ', y_val.shape)

学習データ  (48000, 1, 28, 28)
学習データの正解ラベル  (48000, 10)
検証データ  (12000, 1, 28, 28)
検証データの正解ラベル  (12000, 10)


# 2.モデルの構築

## 必要な基本関数の定義

In [10]:
# シグモイド関数
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# シグモイド関数の微分
def sigmoid_grad(x):
    return (1.0 - sigmoid(x)) * sigmoid(x)

# relu関数
def relu(x):
    return np.maximum(0, x)

# relu関数の微分
def relu_grad(x):
    grad = np.zeros_like(x)
    grad[x>=0] = 1
    return grad

# ソフトマックス関数
def softmax(x):
    x = x - np.max(x, axis=-1, keepdims=True)   # オーバーフロー対策
    return np.exp(x) / np.sum(np.exp(x), axis=-1, keepdims=True)

# 交差エントロピー誤差
def cross_entropy_error(y, t):
    if y.ndim == 1:
        t = t.reshape(1, t.size)
        y = y.reshape(1, y.size)

    # 教師データがone-hot-vectorの場合、正解ラベルのインデックスに変換
    if t.size == y.size:
        t = t.argmax(axis=1)

    batch_size = y.shape[0]
    return -np.sum(np.log(y[np.arange(batch_size), t] + 1e-7)) / batch_size

# ソフトマックス損失
def softmax_loss(X, t):
    y = softmax(X)
    return cross_entropy_error(y, t)

In [11]:
# Relu関数のクラス
class Relu:
    def __init__(self):
        self.mask = None

    def forward(self, x):
        self.mask = (x <= 0)
        out = x.copy()
        out[self.mask] = 0

        return out

    def backward(self, dout):
        dout[self.mask] = 0
        dx = dout

        return dx

# シグモイド関数のクラス
class Sigmoid:
    def __init__(self):
        self.out = None

    def forward(self, x):
        out = sigmoid(x)
        self.out = out
        return out

    def backward(self, dout):
        dx = dout * (1.0 - self.out) * self.out

        return dx

# アフィン層(全結合層)のクラス
class Affine:
    def __init__(self, W, b):
        self.W =W
        self.b = b

        self.x = None
        self.original_x_shape = None
        # 重み・バイアスパラメータの微分
        self.dW = None
        self.db = None

    def forward(self, x):
        # テンソル対応
        self.original_x_shape = x.shape
        x = x.reshape(x.shape[0], -1)
        self.x = x

        out = np.dot(self.x, self.W) + self.b

        return out

    def backward(self, dout):
        dx = np.dot(dout, self.W.T)
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis=0)

        dx = dx.reshape(*self.original_x_shape)  # 入力データの形状に戻す（テンソル対応）
        return dx

# ソフトマックス損失層のクラス
class SoftmaxWithLoss:
    def __init__(self):
        self.loss = None
        self.y = None # softmaxの出力
        self.t = None # 教師データ

    def forward(self, x, t):
        self.t = t
        self.y = softmax(x)
        self.loss = cross_entropy_error(self.y, self.t)

        return self.loss

    def backward(self, dout=1):
        batch_size = self.t.shape[0]
        if self.t.size == self.y.size: # 教師データがone-hot-vectorの場合
            dx = (self.y - self.t) / batch_size
        else:
            dx = self.y.copy()
            dx[np.arange(batch_size), self.t] -= 1
            dx = dx / batch_size

        return dx

# 確率的勾配降下法（Stochastic Gradient Descent)のクラス
class SGD:
    def __init__(self, lr=0.01):
        # 学習率を初期化する（0.01にする）。
        self.lr = lr

    def update(self, params, grads):
        # パラメータの更新を行うメソッド
        # params: 更新されるパラメータ（例：ニューラルネットワークの重みやバイアス）
        # grads: 各パラメータに関する勾配（損失関数の勾配）
        for key in params.keys():
            # 各パラメータに対して、勾配に基づいて更新を行う。
            params[key] -= self.lr * grads[key]

# ドロップアウト層の関数
class Dropout:
    def __init__(self, dropout_ratio=0.5):
        # ドロップアウトの割合を設定する（デフォルトは0.5）
        self.dropout_ratio = dropout_ratio
        self.mask = None

    def forward(self, x, train_flg=True):
        if train_flg:
            # 訓練時は、ランダムにノードをドロップアウトする。
            self.mask = np.random.rand(*x.shape) > self.dropout_ratio
            # マスクを適用して出力を返す。
            return x * self.mask
        else:
            # 評価時は、ドロップアウトは適用せず、
            # 各ノードの出力をドロップアウト比率でスケーリングする。
            return x * (1.0 - self.dropout_ratio)

    def backward(self, dout):
        # 逆伝播時にもマスクを適用する。
        # これにより、前方伝播時にドロップアウトされたノードは、
        # 逆伝播でも影響を与えない。
        return dout * self.mask

# バッチノーマライゼーションのクラス
class BatchNormalization:
    def __init__(self, gamma, beta, momentum=0.9, running_mean=None, running_var=None):
        self.gamma = gamma
        self.beta = beta
        self.momentum = momentum
        self.input_shape = None # Conv層の場合は4次元、全結合層の場合は2次元

        # テスト時に使用する平均と分散
        self.running_mean = running_mean
        self.running_var = running_var

        # backward時に使用する中間データ
        self.batch_size = None
        self.xc = None
        self.std = None
        self.dgamma = None
        self.dbeta = None

    def forward(self, x, train_flg=True):
        self.input_shape = x.shape
        if x.ndim != 2:
            N, C, H, W = x.shape
            x = x.reshape(N, -1)

        out = self.__forward(x, train_flg)

        return out.reshape(*self.input_shape)

    def __forward(self, x, train_flg):
        if self.running_mean is None:
            N, D = x.shape
            self.running_mean = np.zeros(D)
            self.running_var = np.zeros(D)

        if train_flg:
            mu = x.mean(axis=0)
            xc = x - mu
            var = np.mean(xc**2, axis=0)
            std = np.sqrt(var + 10e-7)
            xn = xc / std

            self.batch_size = x.shape[0]
            self.xc = xc
            self.xn = xn
            self.std = std
            self.running_mean = self.momentum * self.running_mean + (1-self.momentum) * mu
            self.running_var = self.momentum * self.running_var + (1-self.momentum) * var
        else:
            xc = x - self.running_mean
            xn = xc / ((np.sqrt(self.running_var + 10e-7)))

        out = self.gamma * xn + self.beta
        return out

    def backward(self, dout):
        if dout.ndim != 2:
            N, C, H, W = dout.shape
            dout = dout.reshape(N, -1)

        dx = self.__backward(dout)

        dx = dx.reshape(*self.input_shape)
        return dx

    def __backward(self, dout):
        dbeta = dout.sum(axis=0)
        dgamma = np.sum(self.xn * dout, axis=0)
        dxn = self.gamma * dout
        dxc = dxn / self.std
        dstd = -np.sum((dxn * self.xc) / (self.std * self.std), axis=0)
        dvar = 0.5 * dstd / self.std
        dxc += (2.0 / self.batch_size) * self.xc * dvar
        dmu = np.sum(dxc, axis=0)
        dx = dxc - dmu / self.batch_size

        self.dgamma = dgamma
        self.dbeta = dbeta

        return dx

In [12]:
# Adam最適化アルゴリズムのクラス
class Adam:
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.iter = 0
        self.m = None
        self.v = None

    def update(self, params, grads):
        if self.m is None:
            self.m, self.v = {}, {}
            for key, val in params.items():
                self.m[key] = np.zeros_like(val)
                self.v[key] = np.zeros_like(val)

        self.iter += 1
        lr_t  = self.lr * np.sqrt(1.0 - self.beta2**self.iter) / (1.0 - self.beta1**self.iter)

        for key in params.keys():
            self.m[key] += (1 - self.beta1) * (grads[key] - self.m[key])
            self.v[key] += (1 - self.beta2) * (grads[key]**2 - self.v[key])

            params[key] -= lr_t * self.m[key] / (np.sqrt(self.v[key]) + 1e-7)

## im2col関数とcol2im関数の実装

In [13]:
# im2col関数
def im2col(input_data, filter_h, filter_w, stride=1, pad=0):
    N, C, H, W = input_data.shape
    out_h = (H + 2*pad - filter_h)//stride + 1
    out_w = (W + 2*pad - filter_w)//stride + 1

    img = np.pad(input_data, [(0,0), (0,0), (pad, pad), (pad, pad)], 'constant')
    col = np.zeros((N, C, filter_h, filter_w, out_h, out_w))

    for y in range(filter_h):
        y_max = y + stride*out_h
        for x in range(filter_w):
            x_max = x + stride*out_w
            col[:, :, y, x, :, :] = img[:, :, y:y_max:stride, x:x_max:stride]

    col = col.transpose(0, 4, 5, 1, 2, 3).reshape(N*out_h*out_w, -1)
    return col

# col2im関数
def col2im(col, input_shape, filter_h, filter_w, stride=1, pad=0):
    N, C, H, W = input_shape
    out_h = (H + 2*pad - filter_h)//stride + 1
    out_w = (W + 2*pad - filter_w)//stride + 1
    col = col.reshape(N, out_h, out_w, C, filter_h, filter_w).transpose(0, 3, 4, 5, 1, 2)

    img = np.zeros((N, C, H + 2*pad + stride - 1, W + 2*pad + stride - 1))
    for y in range(filter_h):
        y_max = y + stride*out_h
        for x in range(filter_w):
            x_max = x + stride*out_w
            img[:, :, y:y_max:stride, x:x_max:stride] += col[:, :, y, x, :, :]

    return img[:, :, pad:H + pad, pad:W + pad]

## Convlution層とプーリング層の実装

In [14]:
# Convlution層のクラス
class Convolution:
    def __init__(self, W, b, stride=1, pad=0):
        self.W = W
        self.b = b
        self.stride = stride
        self.pad = pad

        # 中間データ（backward時に使用）
        self.x = None
        self.col = None
        self.col_W = None

        # 重み・バイアスパラメータの勾配
        self.dW = None
        self.db = None

    def forward(self, x):
        FN, C, FH, FW = self.W.shape
        N, C, H, W = x.shape
        out_h = 1 + int((H + 2*self.pad - FH) / self.stride)
        out_w = 1 + int((W + 2*self.pad - FW) / self.stride)

        col = im2col(x, FH, FW, self.stride, self.pad)
        col_W = self.W.reshape(FN, -1).T

        out = np.dot(col, col_W) + self.b
        out = out.reshape(N, out_h, out_w, -1).transpose(0, 3, 1, 2)

        self.x = x
        self.col = col
        self.col_W = col_W

        return out

    def backward(self, dout):
        FN, C, FH, FW = self.W.shape
        dout = dout.transpose(0,2,3,1).reshape(-1, FN)

        self.db = np.sum(dout, axis=0)
        self.dW = np.dot(self.col.T, dout)
        self.dW = self.dW.transpose(1, 0).reshape(FN, C, FH, FW)

        dcol = np.dot(dout, self.col_W.T)
        dx = col2im(dcol, self.x.shape, FH, FW, self.stride, self.pad)

        return dx

# プーリング層のクラス
class Pooling:
    def __init__(self, pool_h, pool_w, stride=2, pad=0):
        self.pool_h = pool_h
        self.pool_w = pool_w
        self.stride = stride
        self.pad = pad

        self.x = None
        self.arg_max = None

    def forward(self, x):
        N, C, H, W = x.shape
        out_h = int(1 + (H - self.pool_h) / self.stride)
        out_w = int(1 + (W - self.pool_w) / self.stride)

        col = im2col(x, self.pool_h, self.pool_w, self.stride, self.pad)
        col = col.reshape(-1, self.pool_h*self.pool_w)

        arg_max = np.argmax(col, axis=1)
        out = np.max(col, axis=1)
        out = out.reshape(N, out_h, out_w, C).transpose(0, 3, 1, 2)

        self.x = x
        self.arg_max = arg_max

        return out

    def backward(self, dout):
        dout = dout.transpose(0, 2, 3, 1)

        pool_size = self.pool_h * self.pool_w
        dmax = np.zeros((dout.size, pool_size))
        dmax[np.arange(self.arg_max.size), self.arg_max.flatten()] = dout.flatten()
        dmax = dmax.reshape(dout.shape + (pool_size,))

        dcol = dmax.reshape(dmax.shape[0] * dmax.shape[1] * dmax.shape[2], -1)
        dx = col2im(dcol, self.x.shape, self.pool_h, self.pool_w, self.stride, self.pad)

        return dx

## CNNの実装

In [15]:
class DeepConvNet:
    """
    畳み込み 3 層 の軽量版 DeepConvNet
    --------------------------------------------------
    入力 : 1×28×28
       ↓ Conv(16,3×3,s=1,p=1)  … ReLU
       ↓ Conv(16,3×3,s=1,p=1)  … ReLU
       ↓ MaxPool(2×2)          … 14×14
       ↓ Conv(32,3×3,s=1,p=1)  … ReLU
       ↓ MaxPool(2×2)          … 7×7
       ↓ Affine(1568→hidden)   … ReLU
       ↓ Affine(hidden→10)
       ↓ SoftmaxWithLoss
    """
    def __init__(self, input_dim=(1, 28, 28),
                 conv_param_1={'filter_num': 16, 'filter_size': 3, 'pad': 1, 'stride': 1},
                 conv_param_2={'filter_num': 16, 'filter_size': 3, 'pad': 1, 'stride': 1},
                 conv_param_3={'filter_num': 32, 'filter_size': 3, 'pad': 1, 'stride': 1},
                 hidden_size=50, output_size=10):

        # 1. 重み初期化 --------------------------------------------------------
        pre_node_nums = np.array([
            1 * 3 * 3,         # Conv1
            16 * 3 * 3,        # Conv2
            16 * 3 * 3,        # Conv3
            32 * 7 * 7,        # Affine1 入力
            hidden_size        # Affine2 入力
        ])
        weight_scales = np.sqrt(2.0 / pre_node_nums)

        self.params = {}
        pre_channel_num = input_dim[0]
        for idx, conv_param in enumerate([conv_param_1, conv_param_2, conv_param_3], start=1):
            W_key, b_key = f'W{idx}', f'b{idx}'
            self.params[W_key] = weight_scales[idx-1] * np.random.randn(
                conv_param['filter_num'], pre_channel_num,
                conv_param['filter_size'], conv_param['filter_size'])
            self.params[b_key] = np.zeros(conv_param['filter_num'])
            pre_channel_num = conv_param['filter_num']

        self.params['W4'] = weight_scales[3] * np.random.randn(32 * 7 * 7, hidden_size)
        self.params['b4'] = np.zeros(hidden_size)
        self.params['W5'] = weight_scales[4] * np.random.randn(hidden_size, output_size)
        self.params['b5'] = np.zeros(output_size)

        # 2. レイヤ定義 --------------------------------------------------------
        self.layers = [
            # Conv1 → ReLU
            Convolution(self.params['W1'], self.params['b1'],
                        conv_param_1['stride'], conv_param_1['pad']),
            Relu(),
            # Conv2 → ReLU → Pool
            Convolution(self.params['W2'], self.params['b2'],
                        conv_param_2['stride'], conv_param_2['pad']),
            Relu(),
            Pooling(pool_h=2, pool_w=2, stride=2),
            # Conv3 → ReLU → Pool
            Convolution(self.params['W3'], self.params['b3'],
                        conv_param_3['stride'], conv_param_3['pad']),
            Relu(),
            Pooling(pool_h=2, pool_w=2, stride=2),
            # Affine1 → ReLU
            Affine(self.params['W4'], self.params['b4']),
            Relu(),
            # Affine2
            Affine(self.params['W5'], self.params['b5'])
        ]

        self.last_layer = SoftmaxWithLoss()

    # ---------------------------------------------------------------------
    # forward
    def predict(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

    # 損失
    def loss(self, x, t):
        y = self.predict(x)
        return self.last_layer.forward(y, t)

    # 精度
    def accuracy(self, x, t, batch_size=100):
        if t.ndim != 1:
            t = np.argmax(t, axis=1)
        acc = 0.0
        for i in range(int(x.shape[0] / batch_size)):
            tx = x[i*batch_size:(i+1)*batch_size]
            tt = t[i*batch_size:(i+1)*batch_size]
            y = self.predict(tx)
            y = np.argmax(y, axis=1)
            acc += np.sum(y == tt)
        return acc / x.shape[0]

    # 勾配
    def gradient(self, x, t):
        self.loss(x, t)
        dout = 1
        dout = self.last_layer.backward(dout)
        for layer in reversed(self.layers):
            dout = layer.backward(dout)

        # Conv1, Conv2, Conv3, Affine1, Affine2 のインデックス
        trainable_idx = (0, 2, 5, 8, 10)
        grads = {}
        for i, idx in enumerate(trainable_idx, start=1):
            grads[f'W{i}'] = self.layers[idx].dW
            grads[f'b{i}'] = self.layers[idx].db
        return grads

    # パラメータ保存
    def save_params(self, file_name="params.pkl"):
        import pickle
        with open(file_name, 'wb') as f:
            pickle.dump(self.params, f)

    # パラメータ読込
    def load_params(self, file_name="params.pkl"):
        import pickle
        with open(file_name, 'rb') as f:
            self.params = pickle.load(f)
        trainable_idx = (0, 2, 5, 8, 10)
        for i, idx in enumerate(trainable_idx, start=1):
            self.layers[idx].W = self.params[f'W{i}']
            self.layers[idx].b = self.params[f'b{i}']

## 学習用のクラス

In [16]:
# ニューラルネットの訓練を行うクラス
class Trainer:

    def __init__(self, network, x_train, t_train, x_val, t_val,
                 epochs=20, mini_batch_size=100,
                 optimizer='SGD', optimizer_param={'lr':0.01},
                 evaluate_sample_num_per_epoch=None, verbose=True):
        self.network = network
        self.verbose = verbose
        self.x_train = x_train
        self.t_train = t_train
        self.x_val = x_val
        self.t_val = t_val
        self.epochs = epochs
        self.batch_size = mini_batch_size
        self.evaluate_sample_num_per_epoch = evaluate_sample_num_per_epoch

        # optimizer
        optimizer_class_dict = {'sgd':SGD, 'adam':Adam}
        self.optimizer = optimizer_class_dict[optimizer.lower()](**optimizer_param)

        self.train_size = x_train.shape[0]
        self.iter_per_epoch = max(self.train_size / mini_batch_size, 1)
        self.max_iter = int(epochs * self.iter_per_epoch)
        self.current_iter = 0
        self.current_epoch = 0

        self.train_loss_list = []
        self.train_acc_list = []
        self.val_acc_list = []

    def train_step(self):
        batch_mask = np.random.choice(self.train_size, self.batch_size)
        x_batch = self.x_train[batch_mask]
        t_batch = self.t_train[batch_mask]

        grads = self.network.gradient(x_batch, t_batch)
        self.optimizer.update(self.network.params, grads)

        loss = self.network.loss(x_batch, t_batch)
        self.train_loss_list.append(loss)
        if self.verbose: print("train loss:" + str(loss))

        if self.current_iter % self.iter_per_epoch == 0:
            self.current_epoch += 1

            x_train_sample, t_train_sample = self.x_train, self.t_train
            x_val_sample, t_val_sample = self.x_val, self.t_val
            if not self.evaluate_sample_num_per_epoch is None:
                t = self.evaluate_sample_num_per_epoch
                x_train_sample, t_train_sample = self.x_train[:t], self.t_train[:t]
                x_val_sample, t_val_sample = self.x_val[:t], self.t_val[:t]

            train_acc = self.network.accuracy(x_train_sample, t_train_sample)
            val_acc = self.network.accuracy(x_val_sample, t_val_sample)
            self.train_acc_list.append(train_acc)
            self.val_acc_list.append(val_acc)

            if self.verbose: print("=== epoch:" + str(self.current_epoch) + ", train acc:" + str(train_acc) + ", val acc:" + str(val_acc) + " ===")
        self.current_iter += 1

    def train(self):
        for i in range(self.max_iter):
            self.train_step()
            if i % self.iter_per_epoch == 0:
              print(int(i//self.iter_per_epoch)+1, 'epoch')
              val_acc = self.network.accuracy(self.x_val, self.t_val)
              print("val acc:" + str(val_acc))

        val_acc = self.network.accuracy(self.x_val, self.t_val)
        print("=============== 最終 val Accuracy =============")
        print("val acc:" + str(val_acc))

#3.学習から提出まで

## 学習

In [17]:
network = DeepConvNet()
trainer = Trainer(network, x_train, y_train, x_val, y_val,
                  epochs=10,           # epochs remain at 10
                  mini_batch_size=64,  # Reduced mini_batch_size to prevent RAM crash
                  optimizer='Adam',     # Changed optimizer to Adam
                  optimizer_param={'lr':0.001}, # Default learning rate for Adam
                  evaluate_sample_num_per_epoch=1024,
                  verbose=False)
trainer.train()

1 epoch
val acc:0.18633333333333332
3 epoch
val acc:0.9740833333333333
5 epoch
val acc:0.9811666666666666
7 epoch
val acc:0.98725
9 epoch
val acc:0.988
=============== 最終 val Accuracy =============
val acc:0.9875


## テストデータの準備と予測

In [4]:
import locale
import pandas as pd
import numpy as np
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

# テスト用のデータ（正解ラベルなし）のダウンロード
!wget 'https://drive.google.com/uc?export=download&id=18h1BsXvC6q_yZsO_TcONyotw5wXEU9b8' -O mnist_x_test.csv

# テスト用データの確認
x_test = pd.read_csv('mnist_x_test.csv', index_col=0)
x_test.values.shape

--2026-06-11 12:31:16--  https://drive.google.com/uc?export=download&id=18h1BsXvC6q_yZsO_TcONyotw5wXEU9b8
Resolving drive.google.com (drive.google.com)... 192.178.210.138, 192.178.210.101, 192.178.210.113, ...
Connecting to drive.google.com (drive.google.com)|192.178.210.138|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=18h1BsXvC6q_yZsO_TcONyotw5wXEU9b8&export=download [following]
--2026-06-11 12:31:16--  https://drive.usercontent.google.com/download?id=18h1BsXvC6q_yZsO_TcONyotw5wXEU9b8&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 173.194.206.132, 2607:f8b0:4001:c62::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|173.194.206.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 34001360 (32M) [application/octet-stream]
Saving to: ‘mnist_x_test.csv’

mnist_x_test.csv    100%[===================>]  32.43M  

(10000, 784)

In [5]:
# テストデータをDeepConvNetの入力形式に変換
x_test_processed = x_test.values.astype('float32') / 255.0 # 正規化
x_test_processed = np.array([np.expand_dims(img.reshape(28, 28), axis=0) for img in x_test_processed])

print('前処理後のテストデータ形状:', x_test_processed.shape)

前処理後のテストデータ形状: (10000, 1, 28, 28)


In [6]:
# 訓練済みモデルを使用してテストデータで予測を生成
predictions = network.predict(x_test_processed)
predicted_labels = np.argmax(predictions, axis=1)

# ラベルは0から9なので、1から10に変換 (提出形式に合わせる)
predicted_labels_for_submission = predicted_labels + 1

print('予測結果の最初の5つ:', predicted_labels_for_submission[:5])

NameError: name 'network' is not defined

In [ ]:
# 予測結果をDataFrameに格納し、CSV形式で保存
y_pred_submission = pd.DataFrame(predicted_labels_for_submission, columns=['Number'])
y_pred_submission.to_csv('y_pred_final.csv', index=False)

display(y_pred_submission.head())
print("予測結果が 'y_pred_final.csv' として保存されました。")

In [ ]:
# # 提出形式見本
# # 提出データの形、タイトル
# y_pred = pd.DataFrame(np.random.choice(list(range(1, 11)), 10000), columns=['Number'])
# # csv形式で提出して下さい。
# y_pred.to_csv('y_pred.csv')
# y_pred